# FC-Siam-diff on d2c_v3

Training recipe from Daudt et al. (ICIP 2018), `fully-convolutional-change-detection.ipynb`. Model: `siamunet_diff.py` (unmodified). Spec: `low_level_design.md`.

## Config

In [ ]:
from pathlib import Path

DATA_ROOT = Path('../data/processed/d2c_v3').resolve()
INDEX = DATA_ROOT / 'master_dataset_with_benign.json'
SPLITS = DATA_ROOT / 'splits_85_15.json'
RUNS_DIR = Path('../runs')

INPUT_MODE = 'random_squares'  # 'random_squares' | 'all_squares' | 'whole_page'
SQUARE = 96                    # square side in px (notebook PATCH_SIDE)
SQUARES_PER_PAIR = 16          # random_squares: squares cut from each pair per epoch
STRIDE = SQUARE // 2 - 1       # all_squares: step between squares (notebook TRAIN_STRIDE)
PAGE_SIZE = (1024, 512)        # whole_page: (H, W) every page is resized to

BATCH_SIZE = 32
N_EPOCHS = 50
FP_MODIFIER = 10
WEIGHT_DECAY = 1e-4
LR_GAMMA = 0.95
AUG_FLIP = True
AUG_ROT = False

NUM_WORKERS = 12
SEED = 0
BF16 = True
RESUME = None                  # e.g. 'runs/<id>/ckpt_last.pt'

PAIRS_PER_LOAD = {'random_squares': BATCH_SIZE // SQUARES_PER_PAIR, 'all_squares': 1, 'whole_page': BATCH_SIZE}[INPUT_MODE]

In [ ]:
import json, random, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

from siamunet_diff import SiamUnet_diff

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
amp = dict(device_type=device.type, dtype=torch.bfloat16, enabled=BF16 and device.type == 'cuda')
print(device)

## Split by page (`splits_85_15.json`)

In [ ]:
rows = json.loads(INDEX.read_text())
split = {k: set(v) for k, v in json.loads(SPLITS.read_text()).items()}
assert not split['train'] & split['val']

train_rows = [r for r in rows if r['parent_id'] in split['train']]
val_rows = [r for r in rows if r['parent_id'] in split['val']]
assert len(train_rows) + len(val_rows) == len(rows)

for name, rs in (('train', train_rows), ('val', val_rows)):
    print(name, len({r['parent_id'] for r in rs}), 'pages', dict(Counter(r['label'] for r in rs)))

## Pair loading and change mask

Same rule for every pair: decode as RGB, pad both to the larger size with white at the top-left, mark pixels where any channel differs. Regression pairs keep that mask; `no_change` and `benign` get all zeros. Only pixel tensors leave this cell.

In [ ]:
def to_tensor(I):
    I = I.astype(np.float32)
    return torch.from_numpy(((I - I.mean()) / I.std()).transpose(2, 0, 1))

def load_pair(row):
    a = np.asarray(Image.open(DATA_ROOT / row['reference_path']).convert('RGB'))
    b = np.asarray(Image.open(DATA_ROOT / row['candidate_path']).convert('RGB'))
    H, W = max(a.shape[0], b.shape[0]), max(a.shape[1], b.shape[1])
    pa = np.full((H, W, 3), 255, np.uint8); pa[:a.shape[0], :a.shape[1]] = a
    pb = np.full((H, W, 3), 255, np.uint8); pb[:b.shape[0], :b.shape[1]] = b
    mask = (pa != pb).any(2) if row['label'] == 'regression' else np.zeros((H, W), bool)
    return to_tensor(pa), to_tensor(pb), torch.from_numpy(mask)

In [ ]:
# Check: mask size equals the dataset's own changed_px for regressions, zero otherwise
for r in random.sample(train_rows, 30):
    *_, m = load_pair(r)
    assert m.sum() == (r['changed_px'] if r['label'] == 'regression' else 0)
print('masks consistent with index')

## Input modes

In [ ]:
def square_corners(H, W):
    if INPUT_MODE == 'random_squares':
        ys = torch.randint(0, H - SQUARE + 1, (SQUARES_PER_PAIR,))
        xs = torch.randint(0, W - SQUARE + 1, (SQUARES_PER_PAIR,))
        return list(zip(ys.tolist(), xs.tolist()))
    return [(y, x) for y in range(0, H - SQUARE + 1, STRIDE) for x in range(0, W - SQUARE + 1, STRIDE)]

# whole_page: max-pooling the mask keeps every changed pixel visible after downscaling
def to_inputs(I1, I2, mask):
    if INPUT_MODE == 'whole_page':
        resize = lambda I: F.interpolate(I[None], PAGE_SIZE, mode='area')
        return resize(I1), resize(I2), F.adaptive_max_pool2d(mask[None, None].float(), PAGE_SIZE)[0]
    cs = square_corners(*mask.shape)
    crop = lambda t: torch.stack([t[..., y:y + SQUARE, x:x + SQUARE] for y, x in cs])
    return crop(I1), crop(I2), crop(mask)

def augment(I1, I2, m):
    if AUG_FLIP and torch.rand(1) < 0.5:
        I1, I2, m = I1.flip(-1), I2.flip(-1), m.flip(-1)
    if AUG_ROT:
        k = int(torch.randint(0, 4, (1,)))
        I1, I2, m = (t.rot90(k, (-2, -1)) for t in (I1, I2, m))
    return I1, I2, m

## Datasets

In [ ]:
class PairDataset(Dataset):
    def __init__(self, rows, train):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        I1, I2, m = load_pair(self.rows[i])
        if self.train:
            return augment(*to_inputs(I1, I2, m))
        return I1, I2, m, i

def cat_collate(batch):
    return tuple(torch.cat(t) for t in zip(*batch))

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(PairDataset(train_rows, True), batch_size=PAIRS_PER_LOAD, shuffle=True, collate_fn=cat_collate, **loader_kw)
val_loader = DataLoader(PairDataset(val_rows, False), batch_size=1, **loader_kw)

## Network, loss, optimizer

In [ ]:
# Notebook class-weight formula; changed_px equals the mask sum, so no pass over the images is needed
n_pix = sum(max(r['size_ref'][0], r['size_cand'][0]) * max(r['size_ref'][1], r['size_cand'][1]) for r in train_rows)
true_pix = sum(r['changed_px'] for r in train_rows if r['label'] == 'regression')
weights = torch.tensor([FP_MODIFIER * 2 * true_pix / n_pix, 2 * (n_pix - true_pix) / n_pix], device=device)
print(weights)

net = SiamUnet_diff(3, 2).to(device)
criterion = nn.NLLLoss(weight=weights)
optimizer = torch.optim.Adam(net.parameters(), weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, LR_GAMMA)
print('Number of trainable parameters:', sum(p.numel() for p in net.parameters() if p.requires_grad))

## Evaluation (val pages, whole page at once)

In [ ]:
def div(a, b):
    return a / b if b else 0.0

def kappa(tp, tn, fp, fn):
    N = tp + tn + fp + fn
    p0 = (tp + tn) / N
    pe = ((tp + fp) * (tp + fn) + (tn + fp) * (tn + fn)) / (N * N)
    return div(p0 - pe, 1 - pe)

@torch.no_grad()
def evaluate(loader, rows):
    net.eval()
    tp = tn = fp = fn = 0
    tot_loss, flagged = 0.0, {}
    for I1, I2, m, i in tqdm(loader, leave=False):
        I1, I2, m = I1.to(device), I2.to(device), m.to(device)
        if INPUT_MODE == 'whole_page':
            I1, I2, m = to_inputs(I1[0], I2[0], m[0])
        m = m.long()
        with torch.autocast(**amp):
            out = net(I1, I2)
        tot_loss += criterion(out.float(), m).item() * m.numel()
        p, g = out.argmax(1) == 1, m == 1
        tp += (p & g).sum().item(); tn += (~p & ~g).sum().item()
        fp += (p & ~g).sum().item(); fn += (~p & g).sum().item()
        flagged[int(i)] = bool(p.any())

    is_reg = {i: rows[i]['label'] == 'regression' for i in flagged}
    itp = sum(flagged[i] and is_reg[i] for i in flagged)
    ifp = sum(flagged[i] and not is_reg[i] for i in flagged)
    ifn = sum(not flagged[i] and is_reg[i] for i in flagged)
    prec, rec = div(tp, tp + fp), div(tp, tp + fn)
    iprec, irec = div(itp, itp + ifp), div(itp, itp + ifn)
    return {
        'val_loss': tot_loss / (tp + tn + fp + fn),
        'pixel_acc_nochange': div(tn, tn + fp), 'pixel_acc_change': rec,
        'pixel_precision': prec, 'pixel_recall': rec, 'pixel_f1': div(2 * prec * rec, prec + rec),
        'pixel_kappa': kappa(tp, tn, fp, fn),
        'image_accuracy': sum(flagged[i] == is_reg[i] for i in flagged) / len(flagged),
        'image_precision': iprec, 'image_recall': irec, 'image_f1': div(2 * iprec * irec, iprec + irec),
        **{f'image_fpr_{lab}': float(np.mean([flagged[i] for i in flagged if rows[i]['label'] == lab]))
           for lab in ('no_change', 'benign')},
    }

## Train

In [ ]:
def save(path, state):
    tmp = path.with_suffix('.tmp')
    torch.save(state, tmp)
    tmp.replace(path)

start_epoch, best_f1 = 0, -1.0
if RESUME:
    ck = torch.load(RESUME, map_location=device)
    net.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler'])
    start_epoch, best_f1 = ck['epoch'] + 1, ck['best_f1']
    torch.set_rng_state(ck['rng'])
    run_dir = Path(RESUME).parent
else:
    run_dir = RUNS_DIR / time.strftime('%Y%m%d-%H%M%S')
    run_dir.mkdir(parents=True)
    (run_dir / 'config.json').write_text(json.dumps({k: v for k, v in globals().items() if k.isupper()}, default=str, indent=1))
print(run_dir)

In [ ]:
for epoch in range(start_epoch, N_EPOCHS):
    net.train()
    t0, losses = time.time(), []
    for I1, I2, m in tqdm(train_loader, desc=f'epoch {epoch + 1}/{N_EPOCHS}'):
        for s in range(0, len(m), BATCH_SIZE):
            a, b, y = (t[s:s + BATCH_SIZE].to(device, non_blocking=True) for t in (I1, I2, m))
            optimizer.zero_grad()
            with torch.autocast(**amp):
                out = net(a, b)
            loss = criterion(out.float(), y.long())
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
    scheduler.step()

    metrics = {'epoch': epoch + 1, 'train_loss': float(np.mean(losses)), **evaluate(val_loader, val_rows), 'seconds': round(time.time() - t0)}
    print(metrics)
    with open(run_dir / 'metrics.jsonl', 'a') as f:
        f.write(json.dumps(metrics) + '\n')

    is_best = metrics['pixel_f1'] > best_f1
    best_f1 = max(best_f1, metrics['pixel_f1'])
    state = dict(model=net.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                 epoch=epoch, best_f1=best_f1, rng=torch.get_rng_state())
    save(run_dir / 'ckpt_last.pt', state)
    if is_best:
        save(run_dir / 'ckpt_best.pt', state)